# HPO Structure Builder + Organ Mapping + System Mapping

## Purpose
This script constructs a comprehensive mapping framework that bridges **Human Phenotype Ontology (HPO) terms** to **anatomical structures** and **body systems**, enabling downstream anatomical visualization of phenotype distributions using gganatogram.

---

## Pipeline Position

[Previous Stages] → hpo_structure_builder.ipynb → [hpo_to_organ_mapping.ipynb] → [gganatomogram_Plot.Rmd]


---

## Input Files
| File | Description |
|------|-------------|
| `hp.obo` | HPO ontology file (~10 MB, ~20,000 terms) |
| `gganatogram_human_organ_map_*.csv` | List of 78 organs supported by gganatogram |

---

## Output Files
| File | Description | Rows |
|------|-------------|------|
| `hpo_structure_*.csv` | Complete HPO hierarchy (ID, name, depth, parents, ancestors) | 19,393 |
| `organ_phenotype_map_*.csv` | Organ → HPO phenotype mappings | 284 |
| `system_phenotype_map_*.csv` | Body system → HPO phenotype mappings | 147 |
| `combined_phenotype_map_*.csv` | Merged organ + system mappings | 357 |

---

## Methodology Overview

### Part 1: HPO Structure Construction (Sections 1-8)
- Load `hp.obo` using `pronto` library
- Build is_a hierarchy graph (parent ↔ children)
- Calculate depth via BFS from root (HP:0000001)
- Extract all ancestors for each term

### Part 2: Organ-to-Phenotype Mapping (Sections 9-12)
- Filter phenotypes: depth ≤ 5, name contains "Abnormality/Morphology/Physiology"
- Match organ names to phenotype names via text normalization
- Result: 55/78 organs mapped (70.5% coverage)

### Part 3: System-to-Phenotype Mapping (Section 13)
- Define 4 body systems with keyword dictionaries: circulation, nervous_system, respiratory, digestion
- Match phenotypes containing system keywords
- Result: 147 system-phenotype mappings

### Part 4: Merge and Validate (Sections 14-19)
- Full outer join of organ and system mappings on HPO ID
- Validate: no name/depth mismatches
- Result: 357 combined mappings (196 organ-only, 58 system-only, 103 both)

---

## Key Parameters
| Parameter | Value | Description |
|-----------|-------|-------------|
| `HPO_ROOT` | HP:0000001 | Root term for depth calculation |
| `MAX_DEPTH_FOR_MATCHING` | 5 | Maximum phenotype depth for matching |
| `INCLUDE_TERMS` | Abnormality, Morphology, Physiology | Required name substrings |
| `EXCLUDE_TERMS` | History, Exposure, Procedure, ... | Excluded name substrings |

---

## Notes
- Some organ-phenotype mappings were manually curated to complement automated matching
- 23 organs could not be automatically mapped (may require domain expertise)
- Multi-parent HPO terms (3,628) are handled via shortest-path depth calculation

# HPO Structure Builder + Organ Mapping + System Mapping

Build a deterministic CSV file with HPO hierarchy information, then map organs to phenotypes and phenotypes to body systems.

**Input:** `hp.obo`, `gganatogram_human_organ_map_20260127_1002.csv`  
**Output:** `hpo_structure_TIMESTAMP.csv`, `organ_phenotype_map_TIMESTAMP.csv`, `system_phenotype_map_TIMESTAMP.csv`, `combined_phenotype_map_TIMESTAMP.csv`  
**Root:** `HP:0000001` (All)

---
## 1. Configuration

In [ ]:
# =============================================================================
# SCRIPT NAME
# =============================================================================
SCRIPT_NAME = "hpo_structure_builder"

# HPO Root for depth calculation
HPO_ROOT = "HP:0000001"  # All (true root of HPO)

# Depth filter for phenotype matching
MAX_DEPTH_FOR_MATCHING = 5  # Only use phenotypes with depth <= 5

# Organ map file
ORGAN_MAP_FILE = "gganatogram_human_organ_map_20260127_1002.csv"

# =============================================================================
# PHENOTYPE NAME FILTERS
# =============================================================================
# INCLUDE: Must contain at least one of these (case-insensitive)
INCLUDE_TERMS = [
    "Abnormality",
    "Morphology",
    "Physiology"
]

# EXCLUDE: Throw out if contains any of these (case-insensitive)
EXCLUDE_TERMS = [
    "History",
    "Exposure",
    "Procedure",
    "Family history",
    "Biospecimen",
    "Delivery",
    "Speech"
]

# =============================================================================
# BODY SYSTEM KEYWORD DICTIONARIES
# =============================================================================
SYSTEM_KEYWORDS = {
    "circulation": [
        "blood", "vascular", "vasculature", "artery", "arterial",
        "vein", "venous", "capillary", "circulation", "perfusion",
        "hemodynamic", "ischemia", "thrombosis", "embolism",
        "hemorrhage", "cardiac", "coronary", "cerebrovascular"
    ],
    "nervous_system": [
        "brain", "cerebral", "cortical", "nervous", "neuron", "neuronal",
        "neuro", "spinal", "cord", "nerve", "axon", "synapse", "myelin",
        "cerebrospinal", "csf", "neural tube", "ganglion", "neural"
    ],
    "respiratory": [
        "lung", "pulmonary", "respiratory", "airway", "bronchial",
        "bronchus", "alveolar", "trachea", "oxygenation", "ventilation",
        "gas exchange", "hypoxia", "pleura"
    ],
    "digestion": [
        "gastrointestinal", "gi", "digestive", "stomach", "gastric",
        "intestine", "intestinal", "small intestine", "colon", "colonic",
        "bowel", "esophagus", "duodenum", "jejunum", "ileum",
        "liver", "hepatic", "pancreas", "pancreatic", "bile", "biliary",
        "gallbladder"
    ]
}

In [ ]:
# =============================================================================
# Imports
# =============================================================================
import os
import re
from pathlib import Path
from datetime import datetime
from collections import deque, defaultdict

import pronto
import pandas as pd

## 2. Path Definitions

In [ ]:
# =============================================================================
# Directory Structure
# =============================================================================
SCRIPTS_DIR = Path.cwd()
PROJECT_ROOT = SCRIPTS_DIR.parent
INPUT_DIR = PROJECT_ROOT / "Input"
OUTPUT_BASE_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR = OUTPUT_BASE_DIR / SCRIPT_NAME

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if SCRIPTS_DIR.name != "scripts":
    raise RuntimeError(f"Must run from 'scripts' directory. Current: {SCRIPTS_DIR}")

print(f"Input Dir:  {INPUT_DIR}")
print(f"Output Dir: {OUTPUT_DIR}")

In [ ]:
# =============================================================================
# Timestamp Utility
# =============================================================================
def get_timestamp():
    return datetime.now().strftime("%Y%m%d_%H%M")

def get_output_path(filename: str, extension: str = ".csv") -> Path:
    if not extension.startswith("."):
        extension = "." + extension
    return OUTPUT_DIR / f"{filename}_{get_timestamp()}{extension}"

---
## 3. Load hp.obo with Pronto (Safe Parsing)

In [ ]:
# =============================================================================
# Load OBO file using pronto (handles multi-parent, obsolete, alt_id correctly)
# =============================================================================
obo_path = INPUT_DIR / "hp.obo"
print(f"Loading: {obo_path}")
print(f"File size: {obo_path.stat().st_size / 1024 / 1024:.2f} MB")

ont = pronto.Ontology(str(obo_path))

print(f"\nLoaded successfully!")
print(f"Total terms in ontology: {len(ont.terms()):,}")

---
## 4. Filter Terms (Exclude those without valid HP ID)

In [ ]:
# =============================================================================
# Filter: Keep only terms with valid HP: ID
# =============================================================================
all_terms = list(ont.terms())
hp_terms = [t for t in all_terms if t.id.startswith("HP:")]

print(f"All terms: {len(all_terms):,}")
print(f"HP: terms: {len(hp_terms):,}")

# Check for root
if HPO_ROOT not in ont:
    raise ValueError(f"Root term {HPO_ROOT} not found in ontology!")
else:
    root_term = ont[HPO_ROOT]
    print(f"\nRoot term found: {HPO_ROOT} - {root_term.name}")

---
## 5. Build is_a Graph and Calculate Depth (BFS from Root)

In [ ]:
# =============================================================================
# Build is_a-only graph (parent -> children)
# Using ONLY is_a relationships, ignoring 'relationship:' fields
# =============================================================================
children_of = defaultdict(list)
parents_of = defaultdict(list)

for term in hp_terms:
    for parent in term.superclasses(distance=1, with_self=False):
        if parent.id.startswith("HP:"):
            children_of[parent.id].append(term.id)
            parents_of[term.id].append(parent.id)

print(f"Terms with parents: {len(parents_of):,}")
print(f"Terms with children: {len(children_of):,}")

In [ ]:
# =============================================================================
# Calculate depth using BFS (shortest path from root)
# =============================================================================
depth_of = {}
queue = deque([(HPO_ROOT, 0)])
depth_of[HPO_ROOT] = 0

while queue:
    current_id, current_depth = queue.popleft()
    for child_id in children_of[current_id]:
        if child_id not in depth_of:
            depth_of[child_id] = current_depth + 1
            queue.append((child_id, current_depth + 1))

print(f"Terms with depth calculated: {len(depth_of):,}")
assert depth_of[HPO_ROOT] == 0, f"BUG: Root depth should be 0"

In [ ]:
# =============================================================================
# Check for unreachable terms
# =============================================================================
hp_term_ids = {t.id for t in hp_terms}
reachable_from_root = set(depth_of.keys())
unreachable = hp_term_ids - reachable_from_root

print(f"HP terms reachable from {HPO_ROOT}: {len(reachable_from_root):,}")
print(f"HP terms NOT reachable: {len(unreachable):,}")

---
## 6. Extract Ancestors

In [ ]:
# =============================================================================
# Get all ancestors for each term
# =============================================================================
def get_ancestors(term_id: str) -> set:
    ancestors = set()
    queue = deque(parents_of[term_id])
    while queue:
        parent_id = queue.popleft()
        if parent_id not in ancestors:
            ancestors.add(parent_id)
            queue.extend(parents_of[parent_id])
    return ancestors

ancestors_of = {}
for term_id in reachable_from_root:
    ancestors_of[term_id] = get_ancestors(term_id)

print(f"Computed ancestors for {len(ancestors_of):,} terms")

---
## 7. Build DataFrame and Export CSV

In [ ]:
# =============================================================================
# Build output data
# =============================================================================
rows = []
for term in hp_terms:
    term_id = term.id
    if term_id not in depth_of:
        continue
    alt_ids = list(term.alternate_ids) if hasattr(term, 'alternate_ids') else []
    row = {
        'hpo_id': term_id,
        'hpo_name': term.name,
        'depth': depth_of[term_id],
        'is_obsolete': term.obsolete,
        'parent_ids': '|'.join(sorted(parents_of[term_id])),
        'ancestor_ids': '|'.join(sorted(ancestors_of[term_id])),
        'alt_ids': '|'.join(sorted(alt_ids)) if alt_ids else ''
    }
    rows.append(row)

df = pd.DataFrame(rows)
df = df.sort_values('depth').reset_index(drop=True)
print(f"DataFrame shape: {df.shape}")

In [ ]:
# =============================================================================
# Export to CSV
# =============================================================================
output_path = get_output_path("hpo_structure")
df.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")

---
## 8. Sanity Checks

In [ ]:
# =============================================================================
# Quick sanity checks
# =============================================================================
print("=" * 60)
print("SANITY CHECKS")
print("=" * 60)
print(f"Total rows: {len(df):,}")
print(f"Depth range: 0 - {df['depth'].max()}")
print(f"Obsolete terms: {df['is_obsolete'].sum():,}")
obsolete_count = df['is_obsolete'].sum()

# Multi-parent check
df['parent_count'] = df['parent_ids'].apply(lambda x: len(x.split('|')) if x else 0)
multi_parent = df[df['parent_count'] > 1]
print(f"Multi-parent terms: {len(multi_parent):,}")

---
---
# PART 2: Organ to Phenotype Mapping
---

## 9. Filter Phenotypes by Depth and Name

In [ ]:
# =============================================================================
# Filter phenotypes
# =============================================================================
def contains_any(text: str, terms: list) -> bool:
    if not text:
        return False
    text_lower = text.lower()
    return any(term.lower() in text_lower for term in terms)

# Filter 1: depth
df_filtered = df[df['depth'] <= MAX_DEPTH_FOR_MATCHING].copy()
print(f"After depth filter: {len(df_filtered):,}")

# Filter 2: INCLUDE
before_include = len(df_filtered)
df_filtered = df_filtered[df_filtered['hpo_name'].apply(lambda x: contains_any(x, INCLUDE_TERMS))]
print(f"After INCLUDE filter: {len(df_filtered):,}")

# Filter 3: EXCLUDE
before_exclude = len(df_filtered)
df_filtered = df_filtered[~df_filtered['hpo_name'].apply(lambda x: contains_any(x, EXCLUDE_TERMS))]
print(f"After EXCLUDE filter: {len(df_filtered):,}")

## 10. Load Organ Map

In [ ]:
# =============================================================================
# Load organ map file
# =============================================================================
organ_map_path = INPUT_DIR / ORGAN_MAP_FILE
df_organs = pd.read_csv(organ_map_path)
organs = df_organs['organ'].tolist()
print(f"Loaded {len(organs)} organs")

## 11. Flexible Matching Functions

In [ ]:
# =============================================================================
# Matching functions
# =============================================================================
def normalize_text(text: str) -> str:
    if not text:
        return ""
    text = text.lower().replace('_', ' ')
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return ' '.join(text.split())

def get_search_terms(organ: str) -> list:
    normalized = normalize_text(organ)
    terms = [normalized]
    if normalized.endswith('s') and len(normalized) > 3:
        terms.append(normalized[:-1])
    else:
        terms.append(normalized + 's')
    words = normalized.split()
    if len(words) > 1:
        terms.extend(words)
    return terms

def find_matching_phenotypes(organ: str, df_pheno: pd.DataFrame) -> list:
    matches = []
    seen_ids = set()
    search_terms = get_search_terms(organ)
    for _, row in df_pheno.iterrows():
        hpo_name_norm = normalize_text(row['hpo_name'])
        for term in search_terms:
            if term in hpo_name_norm and row['hpo_id'] not in seen_ids:
                match_type = 'exact' if hpo_name_norm == term else 'contains'
                matches.append({
                    'hpo_id': row['hpo_id'],
                    'hpo_name': row['hpo_name'],
                    'depth': row['depth'],
                    'match_type': match_type
                })
                seen_ids.add(row['hpo_id'])
                break
    return matches

## 12. Build Organ-Phenotype Mapping

In [ ]:
# =============================================================================
# Match all organs to phenotypes
# =============================================================================
mapping_rows = []
organs_with_matches = []
organs_without_matches = []

for organ in organs:
    matches = find_matching_phenotypes(organ, df_filtered)
    if matches:
        organs_with_matches.append(organ)
        for match in matches:
            mapping_rows.append({
                'organ': organ,
                'hpo_name': match['hpo_name'],
                'hpo_id': match['hpo_id'],
                'depth': match['depth'],
                'match_type': match['match_type']
            })
    else:
        organs_without_matches.append(organ)

df_mapping = pd.DataFrame(mapping_rows)
print(f"Organs WITH matches: {len(organs_with_matches)}")
print(f"Organs WITHOUT matches: {len(organs_without_matches)}")

In [ ]:
# =============================================================================
# Export organ mapping
# =============================================================================
if len(df_mapping) > 0:
    df_mapping = df_mapping.sort_values(['organ', 'depth']).reset_index(drop=True)
    mapping_output_path = get_output_path("organ_phenotype_map")
    df_mapping.to_csv(mapping_output_path, index=False)
    print(f"Saved: {mapping_output_path}")
else:
    mapping_output_path = None
    print("No organ mappings to save!")

---
---
# PART 3: Body System to Phenotype Mapping
---

## 13. Map Phenotypes to Body Systems

In [ ]:
# =============================================================================
# Map phenotypes to body systems by keywords
# =============================================================================
system_mapping_rows = []

for _, row in df_filtered.iterrows():
    hpo_name_lower = row['hpo_name'].lower()
    for system, keywords in SYSTEM_KEYWORDS.items():
        for keyword in keywords:
            if keyword.lower() in hpo_name_lower:
                system_mapping_rows.append({
                    'system': system,
                    'keyword_matched': keyword,
                    'hpo_name': row['hpo_name'],
                    'hpo_id': row['hpo_id'],
                    'depth': row['depth']
                })
                break

df_system_mapping = pd.DataFrame(system_mapping_rows)
print(f"Total system mappings: {len(df_system_mapping)}")
print(f"Unique phenotypes: {df_system_mapping['hpo_id'].nunique()}")

In [ ]:
# =============================================================================
# Export system mapping
# =============================================================================
if len(df_system_mapping) > 0:
    df_system_mapping = df_system_mapping.sort_values(['system', 'depth']).reset_index(drop=True)
    system_output_path = get_output_path("system_phenotype_map")
    df_system_mapping.to_csv(system_output_path, index=False)
    print(f"Saved: {system_output_path}")
else:
    system_output_path = None
    print("No system mappings to save!")

---
---
# PART 4: Merge Organ and System Mappings
---

## 14. Prepare DataFrames for Merge

In [ ]:
# =============================================================================
# Prepare for merge
# =============================================================================
print("=" * 60)
print("PREPARING DATA FOR MERGE")
print("=" * 60)

df_organ_merge = df_mapping.copy()
df_organ_merge = df_organ_merge.rename(columns={
    'hpo_name': 'hpo_name_organ',
    'depth': 'depth_organ'
})

df_system_merge = df_system_mapping.copy()
df_system_merge = df_system_merge.rename(columns={
    'hpo_name': 'hpo_name_system',
    'depth': 'depth_system'
})

print(f"Organ mapping rows: {len(df_organ_merge)}")
print(f"System mapping rows: {len(df_system_merge)}")
print(f"Organ unique HPO IDs: {df_organ_merge['hpo_id'].nunique()}")
print(f"System unique HPO IDs: {df_system_merge['hpo_id'].nunique()}")

## 15. Perform Full Outer Join

In [ ]:
# =============================================================================
# Full outer join on hpo_id
# =============================================================================
print("=" * 60)
print("MERGING MAPPINGS")
print("=" * 60)

df_combined = pd.merge(
    df_organ_merge,
    df_system_merge,
    on='hpo_id',
    how='outer',
    indicator=True
)

merge_counts = df_combined['_merge'].value_counts()
print(f"\nMerge results:")
print(f"  Only in organ_map:  {merge_counts.get('left_only', 0):,}")
print(f"  Only in system_map: {merge_counts.get('right_only', 0):,}")
print(f"  In BOTH:            {merge_counts.get('both', 0):,}")
print(f"  Total rows:         {len(df_combined):,}")

## 16. Check for Mismatches

In [ ]:
# =============================================================================
# Check for mismatches in rows that appear in both
# =============================================================================
print("=" * 60)
print("MISMATCH DETECTION")
print("=" * 60)

both_rows = df_combined[df_combined['_merge'] == 'both'].copy()

# Check hpo_name mismatches
name_mismatches = both_rows[
    both_rows['hpo_name_organ'] != both_rows['hpo_name_system']
]

print(f"\n1. HPO NAME MISMATCHES: {len(name_mismatches)}")
if len(name_mismatches) > 0:
    print("   WARNING! Different names for same HPO ID:")
    for _, row in name_mismatches.head(5).iterrows():
        print(f"   {row['hpo_id']}:")
        print(f"     Organ:  {row['hpo_name_organ']}")
        print(f"     System: {row['hpo_name_system']}")
else:
    print("   OK - All names match!")

# Check depth mismatches
depth_mismatches = both_rows[
    both_rows['depth_organ'] != both_rows['depth_system']
]

print(f"\n2. DEPTH MISMATCHES: {len(depth_mismatches)}")
if len(depth_mismatches) > 0:
    print("   WARNING! Different depths for same HPO ID:")
    for _, row in depth_mismatches.head(5).iterrows():
        print(f"   {row['hpo_id']}: organ={row['depth_organ']}, system={row['depth_system']}")
else:
    print("   OK - All depths match!")

## 17. Create Final Combined Table

In [ ]:
# =============================================================================
# Create unified columns
# =============================================================================
df_combined['hpo_name'] = df_combined['hpo_name_organ'].fillna(df_combined['hpo_name_system'])
df_combined['depth'] = df_combined['depth_organ'].fillna(df_combined['depth_system']).astype(int)

df_combined['source'] = df_combined['_merge'].map({
    'left_only': 'organ_only',
    'right_only': 'system_only',
    'both': 'both'
})

# Select final columns
final_columns = [
    'hpo_id', 'hpo_name', 'depth',
    'organ', 'match_type',
    'system', 'keyword_matched',
    'source'
]

df_final = df_combined[final_columns].copy()
print(f"Final combined table: {len(df_final)} rows")
df_final.head(10)

## 18. Combined Mapping Validation

In [ ]:
# =============================================================================
# Validation: Source distribution
# =============================================================================
print("=" * 60)
print("VALIDATION: SOURCE DISTRIBUTION")
print("=" * 60)

source_counts = df_final['source'].value_counts()
print(f"\n{'Source':<15} {'Count':>10} {'Percent':>10}")
print("-" * 37)
for source, count in source_counts.items():
    pct = count / len(df_final) * 100
    print(f"{source:<15} {count:>10,} {pct:>9.1f}%")

# Sample of 'both' rows
both_source = df_final[df_final['source'] == 'both']
print(f"\nSample rows with BOTH organ and system ({len(both_source)} total):")
for _, row in both_source.head(5).iterrows():
    print(f"  {row['hpo_id']}: {row['hpo_name'][:40]}")
    print(f"    organ={row['organ']}, system={row['system']}")

## 19. Export Combined Mapping

In [ ]:
# =============================================================================
# Export combined mapping
# =============================================================================
df_final = df_final.sort_values(['source', 'depth', 'hpo_name']).reset_index(drop=True)

combined_output_path = get_output_path("combined_phenotype_map")
df_final.to_csv(combined_output_path, index=False)

print(f"Saved combined mapping to: {combined_output_path}")
print(f"File size: {combined_output_path.stat().st_size / 1024:.1f} KB")
print(f"Total rows: {len(df_final)}")

In [ ]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "=" * 60)
print("FINAL SUMMARY - ALL OUTPUTS")
print("=" * 60)
print(f"""
INPUT:
  - HPO ontology: {len(df):,} terms
  - Filtered phenotypes: {len(df_filtered):,} terms
  - Organs: {len(organs)}
  - Body systems: {len(SYSTEM_KEYWORDS)}

OUTPUT FILES:
  1. {output_path.name}
     HPO structure: {len(df):,} terms
     
  2. {mapping_output_path.name if mapping_output_path else 'N/A'}
     Organ mappings: {len(df_mapping):,} rows
     Organs mapped: {len(organs_with_matches)} / {len(organs)}
     
  3. {system_output_path.name if system_output_path else 'N/A'}
     System mappings: {len(df_system_mapping):,} rows
     
  4. {combined_output_path.name}
     Combined mappings: {len(df_final):,} rows
     - Organ only: {source_counts.get('organ_only', 0):,}
     - System only: {source_counts.get('system_only', 0):,}
     - Both: {source_counts.get('both', 0):,}

MISMATCH REPORT:
  - Name mismatches: {len(name_mismatches)}
  - Depth mismatches: {len(depth_mismatches)}
""")